# Token Classification with Transformers

This notebook demonstrates how to build an end-to-end token classification pipeline using the Hugging Face Transformers library. Token classification assigns a label to each token in a sentence and is commonly used for tasks such as Named Entity Recognition (NER).

## Learning Objectives

- Understand token classification tasks
- Load and explore NER datasets
- Tokenize text while preserving label alignment
- Build data collators for token classification
- Fine-tune Transformer models
- Evaluate token classification models

## Technologies

- Python
- PyTorch
- Hugging Face Transformers
- Hugging Face Datasets
- Hugging Face Evaluate
- Accelerate

You will need to setup git, adapt your email and name in the following cell.

In [1]:
!git config --global user.email "milad.saeedi@mail.utoronto.ca"
!git config --global user.name "Milad Saeedi"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

## Loading the Dataset

The first step is to load a labeled Named Entity Recognition (NER) dataset. Unlike sentence classification tasks, token classification datasets contain a sequence of words together with a label assigned to each individual token.

In [2]:
from datasets import load_dataset

raw_datasets = load_dataset(
    "conll2003",
    trust_remote_code=True
)

In [3]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [4]:


raw_datasets["train"] = (
    raw_datasets["train"]
    .shuffle(seed=42)
    .select(range(1000))
)

raw_datasets["validation"] = (
    raw_datasets["validation"]
    .shuffle(seed=42)
    .select(range(200))
)

raw_datasets["test"] = (
    raw_datasets["test"]
    .shuffle(seed=42)
    .select(range(200))
)

### Exploring the Dataset

Before training a model, it is useful to inspect the dataset structure. In this section, we examine the input tokens and their corresponding NER labels to better understand how the data is organized.

In [5]:
raw_datasets["train"][0]["tokens"]

['"',
 'Neither',
 'the',
 'National',
 'Socialists',
 '(',
 'Nazis',
 ')',
 'nor',
 'the',
 'communists',
 'dared',
 'to',
 'kidnap',
 'an',
 'American',
 'citizen',
 ',',
 '"',
 'he',
 'shouted',
 ',',
 'in',
 'an',
 'oblique',
 'reference',
 'to',
 'his',
 'extradition',
 'to',
 'Germany',
 'from',
 'Denmark',
 '.',
 '"']

In [6]:
raw_datasets["train"][0]["ner_tags"]

[0,
 0,
 0,
 7,
 8,
 0,
 7,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 7,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 5,
 0,
 5,
 0,
 0]

In [7]:
ner_feature = raw_datasets["train"].features["ner_tags"]
ner_feature

Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)

In [8]:
label_names = ner_feature.feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [9]:
words = raw_datasets["train"][0]["tokens"]
labels = raw_datasets["train"][0]["ner_tags"]
line1 = ""
line2 = ""
for word, label in zip(words, labels):
    full_label = label_names[label]
    max_length = max(len(word), len(full_label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += full_label + " " * (max_length - len(full_label) + 1)

print(line1)
print(line2)

" Neither the National Socialists ( Nazis  ) nor the communists dared to kidnap an American citizen , " he shouted , in an oblique reference to his extradition to Germany from Denmark . " 
O O       O   B-MISC   I-MISC     O B-MISC O O   O   O          O     O  O      O  B-MISC   O       O O O  O       O O  O  O       O         O  O   O           O  B-LOC   O    B-LOC   O O 


## Tokenizing the Dataset

Transformer models operate on subword tokens rather than complete words. This section demonstrates how to tokenize the input text while preserving the relationship between the original words and their corresponding labels.

In [10]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [11]:
tokenizer.is_fast

True

In [12]:
inputs = tokenizer(raw_datasets["train"][0]["tokens"], is_split_into_words=True)
inputs.tokens()

['[CLS]',
 '"',
 'Neither',
 'the',
 'National',
 'Socialist',
 '##s',
 '(',
 'Nazis',
 ')',
 'nor',
 'the',
 'communists',
 'dared',
 'to',
 'kidnap',
 'an',
 'American',
 'citizen',
 ',',
 '"',
 'he',
 'shouted',
 ',',
 'in',
 'an',
 'oblique',
 'reference',
 'to',
 'his',
 'extra',
 '##dition',
 'to',
 'Germany',
 'from',
 'Denmark',
 '.',
 '"',
 '[SEP]']

In [13]:
inputs.word_ids()

[None,
 0,
 1,
 2,
 3,
 4,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 None]

### Aligning Labels with Subword Tokens

A single word may be split into multiple subword tokens during tokenization. Since the original dataset provides one label per word, we must align these labels with the generated subword tokens before training.

In [14]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            # Special token
            new_labels.append(-100)
        else:
            # Same word as previous token
            label = labels[word_id]
            # If the label is B-XXX we change it to I-XXX
            if label % 2 == 1:
                label += 1
            new_labels.append(label)

    return new_labels

### Preprocessing the Dataset

After defining the alignment procedure, we apply it to every example in the dataset. This produces tokenized inputs together with correctly aligned labels that are ready for model training.

In [15]:
labels = raw_datasets["train"][0]["ner_tags"]
word_ids = inputs.word_ids()
print(labels)
print(align_labels_with_tokens(labels, word_ids))

[0, 0, 0, 7, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0]
[-100, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, -100]


In [16]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [17]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Dynamic Padding

Training batches often contain sequences of different lengths. The data collator dynamically pads each batch while ensuring that token labels remain correctly aligned with the input tokens.

In [18]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [19]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(2)])
batch["labels"]

tensor([[-100,    0,    0,    0,    7,    8,    8,    0,    7,    0,    0,    0,
            0,    0,    0,    0,    0,    7,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    5,    0,    5,
            0,    0, -100],
        [-100,    5,    6,    6,    0,    0,    0,    0,    0, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100]])

In [20]:
for i in range(2):
    print(tokenized_datasets["train"][i]["labels"])

[-100, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, -100]
[-100, 5, 6, 6, 0, 0, 0, 0, 0, -100]


## Evaluation Metrics

Token classification models are commonly evaluated using precision, recall, and F1-score. In this notebook, we use the `seqeval` library to compute these metrics for Named Entity Recognition.

In [21]:
!pip install seqeval

In [22]:
import evaluate

metric = evaluate.load("seqeval")

In [23]:
labels = raw_datasets["train"][0]["ner_tags"]
labels = [label_names[i] for i in labels]
labels

['O',
 'O',
 'O',
 'B-MISC',
 'I-MISC',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-MISC',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-LOC',
 'O',
 'B-LOC',
 'O',
 'O']

In [24]:
predictions = labels.copy()
predictions[2] = "O"
metric.compute(predictions=[predictions], references=[labels])

{'LOC': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(2)},
 'MISC': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(3)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

In [25]:
import numpy as np


def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [26]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [27]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
model.config.num_labels

9

## Configuring Model Training

The `TrainingArguments` class defines the training configuration, including learning rate, batch size, evaluation strategy, checkpointing, and logging behavior.

In [31]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-ner-tokenclass",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

### Fine-Tuning with the Trainer API

The Hugging Face `Trainer` API manages the complete training workflow, including optimization, evaluation, checkpointing, and metric computation.

In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)
trainer.train()

/var/folders/nn/c8hngzbx03v9v7qk14mjl0680000gn/T/ipykernel_56395/3203677919.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.121676,0.780549,0.857534,0.817232,0.967813
2,No log,0.096754,0.839378,0.887671,0.862850,0.971253
3,No log,0.091997,0.845361,0.898630,0.871182,0.974201


TrainOutput(global_step=375, training_loss=0.13316387939453125, metrics={'train_runtime': 551.0244, 'train_samples_per_second': 5.444, 'train_steps_per_second': 0.681, 'total_flos': 65940776359200.0, 'train_loss': 0.13316387939453125, 'epoch': 3.0})

In [33]:
trainer.push_to_hub(commit_message="Training complete")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Miladsaeedi70/bert-finetuned-ner-tokenclass/commit/7466e8a79d5ee7837b86f89ea03ddee224d20651', commit_message='Training complete', commit_description='', oid='7466e8a79d5ee7837b86f89ea03ddee224d20651', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Miladsaeedi70/bert-finetuned-ner-tokenclass', endpoint='https://huggingface.co', repo_type='model', repo_id='Miladsaeedi70/bert-finetuned-ner-tokenclass'), pr_revision=None, pr_num=None)

## Manual Training Loop with Accelerate

In addition to the high-level `Trainer` API, Hugging Face provides the `Accelerate` library for implementing custom training loops. This approach offers greater flexibility while simplifying multi-device training.

In [34]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [35]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [36]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [37]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [38]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [39]:
from huggingface_hub import Repository, get_full_repo_name

model_name = "bert-finetuned-ner-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'Miladsaeedi70/bert-finetuned-ner-accelerate'

In [ ]:
output_dir = "bert-finetuned-ner-accelerate"
repo = Repository(output_dir, clone_from=repo_name)

In [41]:
def postprocess(predictions, labels):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

In [42]:
output_dir

'bert-finetuned-ner-accelerate'

In [ ]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for batch in eval_dataloader:
        with torch.no_grad():
            outputs = model(**batch)

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]

        # Necessary to pad predictions and labels for being gathered
        predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(predictions)
        labels_gathered = accelerator.gather(labels)

        true_predictions, true_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=true_predictions, references=true_labels)

    results = metric.compute()
    print(
        f"epoch {epoch}:",
        {
            key: results[f"overall_{key}"]
            for key in ["precision", "recall", "f1", "accuracy"]
        },
    )


In [ ]:
accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "huggingface-course/bert-finetuned-ner"
token_classifier = pipeline(
    "token-classification", model=model_checkpoint, aggregation_strategy="simple"
)
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[{'entity_group': 'PER', 'score': 0.9988506, 'word': 'Sylvain', 'start': 11, 'end': 18},
 {'entity_group': 'ORG', 'score': 0.9647625, 'word': 'Hugging Face', 'start': 33, 'end': 45},
 {'entity_group': 'LOC', 'score': 0.9986118, 'word': 'Brooklyn', 'start': 49, 'end': 57}]

---

# Key Takeaways

In this notebook:

- Load and explore Named Entity Recognition datasets.
- Tokenize text while preserving word-label alignment.
- Handle subword tokenization for token classification.
- Dynamically pad batches for efficient training.
- Fine-tune pretrained Transformer models for token classification.
- Evaluate NER models using precision, recall, and F1-score.
- Implement both high-level (`Trainer`) and custom (`Accelerate`) training workflows.